# Bellwether — strategy detection

Accounts are what Polymarket exposes; **strategies** are the durable signal — a
behavioral template run over and over: across markets in a recurring family
(scalping `btc-updown-5m-*`), across accounts (same archetype again and again),
or as copy/follow chains.

1. behavioral features → 2. explainable archetype → 3. repeated templates →
4. lead-lag copy chains.

In [ ]:
from bellwether_analytics.core import load_events_df, load_trades_df
from bellwether_analytics.strategy import (
    classify,
    detect_followers,
    extract_features,
    strategy_templates,
)

trades = load_trades_df(platform="polymarket")
events = load_events_df(platform="polymarket")
feats = extract_features(trades, events)
labeled = classify(feats)
labeled[["archetype", "trades_per_day", "net_direction", "roundtrip_ratio",
         "split_merge_ratio", "top_family", "top_family_share", "reason"]]

In [ ]:
# Strategies implemented over and over: (archetype, market-family) across accounts.
strategy_templates(trades, labeled["archetype"], min_wallets=2)

In [ ]:
# Copy/follow chains: wallets that systematically trade just after another.
detect_followers(trades, max_lag_seconds=120, min_events=3)

## Temporal recurrence (same cycle, over and over in time)
Does a wallet run the same buy→accumulate→redeem cycle on a regular cadence?
High regularity + many completed cycles = a templated strategy run repeatedly.

In [ ]:
import pandas as pd
from bellwether_analytics.strategy import recurrence_in_time_report

rows = [recurrence_in_time_report(trades, events, w) for w in trades['wallet'].unique()[:5]]
pd.DataFrame(rows)[['wallet','top_family','is_recurring','regularity','n_cycles',
                    'dominant_period_seconds','cycle_consistency']]

## Strategy -> profitability (templates, not accounts)
Rank (archetype, market-family) templates by strictly out-of-sample realized
P&L. The shuffled-label control must show no edge.

In [ ]:
from bellwether_analytics.strategy import rank_templates

split = '2026-05-01'
ranked = rank_templates(trades, split_ts=split)
shuffled = rank_templates(trades, split_ts=split, shuffle=True, seed=0)
display(ranked.head(15))
print('shuffled-control top median P&L:', None if shuffled.empty else shuffled.iloc[0]['median_pnl'])

## Real-data trackability (Experiment A from real followers)
For significant copy-chains, what did real followers actually capture?

In [ ]:
from bellwether_analytics.experiments import rank_leaders
from bellwether_analytics.strategy import detect_followers_significant

pairs = detect_followers_significant(trades, max_lag_seconds=120, min_events=3)
rank_leaders(trades, pairs)  # per-leader empirical copyability + captured edge

## Unsupervised cross-check of the rule archetypes (diagnostic)
Cluster the feature vectors; compare to rule labels; surface disagreements.

In [ ]:
from bellwether_analytics.strategy import cluster_wallets, compare_to_rules, recommend_thresholds

clusters = cluster_wallets(feats, k=3)
cmp = compare_to_rules(clusters, labeled['archetype'])
print('agreement:', cmp['agreement'])
display(cmp['contingency'])
recommend_thresholds(feats, cmp)

## Trustworthy copy-chains: null model + on-chain block-gap confirmation
On the full pool, synchronized reaction to public news looks like copying.
The guard is two-stage: permutation null model (timestamps) THEN on-chain
block-gap confirmation (the follower must land a small, CONSISTENT block gap
after the leader, repeatedly). Then measure what real followers captured.

In [ ]:
from bellwether_analytics.experiments import trackability_verdict
from bellwether_analytics.strategy import confirmed_copy_chains

chains = confirmed_copy_chains(trades, max_lag_seconds=120, min_events=3)
display(chains)  # significance-filtered + block-confirmed, with block-gap profile
trackability_verdict(trades, max_lag_seconds=120, min_events=3)  # per-leader capture

## Feasible strategy templates: per-archetype P&L boxplots + ranked table
Out-of-sample realized P&L by archetype, and the ranked (archetype, family)
templates with win rate, consistency (Sharpe), distinct-wallet count,
persistence across sub-periods, and a capacity proxy (P&L vs position size).
The shuffled-label control shows the ranking is not an artifact.

In [ ]:
import matplotlib.pyplot as plt
from bellwether_analytics.core import performance_by_wallet
from bellwether_analytics.strategy import classify, extract_features, rank_templates

split = '2026-05-01'
ts = pd.to_datetime(trades['ts'], utc=True)
train, test = trades[ts < split], trades[ts >= split]
labels = classify(extract_features(train))['archetype']
oos_pnl = performance_by_wallet(test).get('realized_pnl')
box = pd.DataFrame({'archetype': labels, 'pnl': oos_pnl}).dropna(subset=['archetype'])
box['pnl'] = box['pnl'].fillna(0.0)
if not box.empty:
    ax = box.boxplot(column='pnl', by='archetype')
    ax.set_title('Per-archetype realized P&L (out-of-sample)')
    ax.set_ylabel('realized P&L')
    plt.suptitle('')
    plt.show()


In [ ]:
# Ranked FEASIBLE templates (out-of-sample) + shuffled-label control.
ranked = rank_templates(trades, split_ts=split, n_periods=3)
cols = ['archetype','family','n_wallets','median_pnl','pct_profitable','sharpe',
        'persistence','pnl_per_volume','capacity_corr']
display(ranked[cols] if not ranked.empty else ranked)
shuffled = rank_templates(trades, split_ts=split, n_periods=3, shuffle=True, seed=0)
print('shuffled-control top median P&L:', None if shuffled.empty else shuffled.iloc[0]['median_pnl'])